# JSON5Kit Parser Testing

In [ ]:
# Experimental setup
import random
import time
from statistics import mean, median

import json5kit
import coverage

from fuzzingbook.Fuzzer import RandomFuzzer
from fuzzingbook.GrammarCoverageFuzzer import TrackingGrammarCoverageFuzzer
from fuzzingbook.ProbabilisticGrammarFuzzer import ProbabilisticGrammarFuzzer, ExpansionCountMiner
from fuzzingbook.Parser import EarleyParser
from fuzzingbook.GrammarFuzzer import all_terminals
from fuzzingbook.Grammars import crange, extend_grammar, set_opts, opts
from fuzzingbook.MutationFuzzer import MutationFuzzer

SEED = 42
INPUT_COUNTS = [2500, 5000, 7500, 10000]
MAX_NONTERMINALS = 10
OBJECT_PROBABILITIES = [0.25, 0.40, 0.55]
MUTATION_SETTINGS = [(1, 3), (2, 5), (5, 10)]


In [ ]:
# Random fuzzing
random_inputs = {}
generation_times = {}

for n in INPUT_COUNTS:
    random.seed(SEED)
    random_fuzzer = RandomFuzzer(min_length=1, max_length=100,
                                 char_start=32, char_range=95)

    start = time.perf_counter()
    random_inputs[n] = [random_fuzzer.fuzz() for _ in range(n)]
    generation_times['Random', n] = time.perf_counter() - start

    print("\nRANDOM FUZZER -", n, "INPUTS")
    print("Generated inputs:", len(random_inputs[n]))
    for i, text in enumerate(random_inputs[n][:5], 1):
        print(f"Input {i}: {text!r}")

In [ ]:
code_coverage = {}
parser_results = {}

# Random parser code coverage
for n, inputs in random_inputs.items():
    cov = coverage.Coverage(source=["json5kit"], branch=False, data_file=None)
    accepted = 0
    rejected = 0
    unexpected = 0

    cov.start()
    try:
        for test_input in inputs:
            try:
                json5kit.parse(test_input)
                accepted += 1
            except (ValueError, SyntaxError):
                rejected += 1
            except Exception as error:
                unexpected += 1
                print(type(error).__name__, error)
    finally:
        cov.stop()

    print("\nRANDOM FUZZER - JSON5KIT CODE COVERAGE")
    print("Input count:", n)
    code_coverage['Random', n] = cov.report()
    parser_results['Random', n] = (accepted, rejected, unexpected)

    print(f"\nTotal line coverage: {code_coverage['Random', n]:.2f}%")
    print("Accepted:", accepted)
    print("Rejected:", rejected)
    print("Unexpected errors:", unexpected)
    print(f"Acceptance rate: {accepted / n * 100:.2f}%")

In [ ]:
# JSON5 grammar construction
JSON5_GRAMMAR = {
    "<start>": ["<value>"],

    "<value>": [
        "<object>", "<array>", "<string>", "<number>",
        "true", "false", "null"
    ],

    "<object>": [
        "{}", "{<members>}", "{<members>,}",
        "{ <members> }", "{<space><members><space>}",
        "{//comment\n<members>}"
    ],
    "<members>": [
        "<pair>", "<pair>, <members>",
        "<pair>,<space><members>"
    ],
    "<pair>": ["<key>: <value>", "<key>:<value>"],
    "<key>": ["<identifier>", "<string>"],

    "<identifier>": [
        "<id_start>", "<id_start><id_part>",
        "<id_start><id_part><id_part>",
        "<id_start><id_part><id_part><id_part>"
    ],
    "<id_start>": crange("a", "z") + crange("A", "Z") + ["_", "$"],
    "<id_part>": crange("a", "z") + crange("A", "Z") + crange("0", "9") + ["_", "$"],

    "<array>": [
        "[]", "[<elements>]", "[<elements>,]",
        "[ <elements> ]", "[<space><elements><space>]"
    ],
    "<elements>": [
        "<value>", "<value>, <elements>",
        "<value>,<space><elements>"
    ],

    "<string>": [
        '""', "''", '"<chars>"', "'<chars>'",
        '"<chars>\\n<chars>"', "'<chars>\\t<chars>'"
    ],
    "<chars>": [
        "<char>", "<char><char>", "<char><char><char>",
        "<char><char><char><char>"
    ],
    "<char>": crange("a", "z") + crange("A", "Z") + crange("0", "9") + [" ", "_", "-", "."],

    "<number>": [
        "<integer>", "<integer>.<digits>",
        "<integer>e<exp_sign><digits>", "0x<hex_digits>",
        "+<unsigned_integer>", "<integer>.",
        ".<digits>", "-.<digits>",
        "Infinity", "-Infinity", "NaN"
    ],
    "<integer>": ["0", "<unsigned_integer>", "-<unsigned_integer>"],
    "<unsigned_integer>": [
        "<nonzero>", "<nonzero><digit>",
        "<nonzero><digit><digit>"
    ],
    "<nonzero>": crange("1", "9"),
    "<digit>": crange("0", "9"),
    "<digits>": ["<digit>", "<digit><digit>", "<digit><digit><digit>"],
    "<hex_digits>": ["<hex_digit>", "<hex_digit><hex_digit>", "<hex_digit><hex_digit><hex_digit>"],
    "<hex_digit>": crange("0", "9") + crange("a", "f") + crange("A", "F"),
    "<exp_sign>": ["", "+", "-"],
    "<space>": [" ", "\n", "\t"]
}

print('Grammar non-terminals:', len(JSON5_GRAMMAR))

In [ ]:
# Normal grammar fuzzing
normal_inputs = {}
normal_used = {}
all_productions = TrackingGrammarCoverageFuzzer(JSON5_GRAMMAR).max_expansion_coverage()

for n in INPUT_COUNTS:
    random.seed(SEED)
    normal_fuzzer = TrackingGrammarCoverageFuzzer(
        JSON5_GRAMMAR, max_nonterminals=MAX_NONTERMINALS
    )

    start = time.perf_counter()
    normal_inputs[n] = [normal_fuzzer.fuzz() for _ in range(n)]
    generation_times['Normal', n] = time.perf_counter() - start
    normal_used[n] = set(normal_fuzzer.expansion_coverage())

    print("\nNORMAL GRAMMAR FUZZER -", n, "INPUTS")
    print("Generated inputs:", len(normal_inputs[n]))
    for i, text in enumerate(normal_inputs[n][:5], 1):
        print(f"Input {i}: {text}")

In [ ]:
# Normal generated input testing
for n, inputs in normal_inputs.items():
    accepted = 0
    rejected = 0
    unexpected = 0

    print("\nNORMAL GENERATED INPUT TESTING -", n, "INPUTS")
    for i, text in enumerate(inputs, 1):
        try:
            json5kit.parse(text)
            accepted += 1
            status = "ACCEPTED"
        except (ValueError, SyntaxError):
            rejected += 1
            status = "REJECTED"
        except Exception as error:
            unexpected += 1
            status = "UNEXPECTED ERROR"
            print(type(error).__name__, error)

        if i <= 5:
            print(f"Test {i}: {status}")
            print(f"Input: {text}")

    print("\n--- TEST SUMMARY ---")
    print("Total inputs:", n)
    print("Accepted:", accepted)
    print("Rejected:", rejected)
    print("Unexpected errors:", unexpected)
    print(f"Acceptance rate: {accepted / n * 100:.2f}%")

In [ ]:
# Normal grammar production coverage
normal_grammar_coverage = {}
for n in INPUT_COUNTS:
    used = normal_used[n]
    normal_grammar_coverage[n] = len(used) / len(all_productions) * 100

    print("\nNORMAL GRAMMAR COVERAGE -", n, "INPUTS")
    print("Total grammar productions:", len(all_productions))
    print("Used productions:", len(used))
    print(f"Grammar coverage: {normal_grammar_coverage[n]:.2f}%")
    print("Missing productions:", sorted(all_productions - used))

In [ ]:
# Normal json5kit code coverage
for n, inputs in normal_inputs.items():
    cov = coverage.Coverage(source=["json5kit"], branch=False, data_file=None)
    accepted = 0
    rejected = 0
    unexpected = 0

    cov.start()
    try:
        for test_input in inputs:
            try:
                json5kit.parse(test_input)
                accepted += 1
            except (ValueError, SyntaxError):
                rejected += 1
            except Exception as error:
                unexpected += 1
                print(type(error).__name__, error)
    finally:
        cov.stop()

    print("\nNORMAL FUZZER - JSON5KIT CODE COVERAGE")
    print("Input count:", n)
    code_coverage['Normal', n] = cov.report()
    parser_results['Normal', n] = (accepted, rejected, unexpected)

    print(f"\nTotal line coverage: {code_coverage['Normal', n]:.2f}%")
    print("Accepted:", accepted)
    print("Rejected:", rejected)
    print("Unexpected errors:", unexpected)
    print(f"Acceptance rate: {accepted / n * 100:.2f}%")

In [ ]:
# Heuristic-guided grammar fuzzer
heuristic_grammars = {}
for probability in OBJECT_PROBABILITIES:
    guided_grammar = extend_grammar(JSON5_GRAMMAR)
    set_opts(guided_grammar, '<value>', '<object>', opts(prob=probability))
    heuristic_grammars[probability] = guided_grammar
    print("Object probability:", probability)

In [ ]:
# Heuristic input generation
heuristic_inputs = {}
heuristic_used = {}

for probability in OBJECT_PROBABILITIES:
    for n in INPUT_COUNTS:
        random.seed(SEED)
        guided_fuzzer = ProbabilisticGrammarFuzzer(
            heuristic_grammars[probability], max_nonterminals=MAX_NONTERMINALS
        )
        grammar_miner = ExpansionCountMiner(EarleyParser(JSON5_GRAMMAR))

        inputs = []
        generation_time = 0
        for _ in range(n):
            start = time.perf_counter()
            tree = guided_fuzzer.fuzz_tree()
            text = all_terminals(tree)
            generation_time += time.perf_counter() - start
            inputs.append(text)
            grammar_miner.add_tree(tree)

        heuristic_inputs[probability, n] = inputs
        heuristic_used[probability, n] = set(grammar_miner.counts())
        generation_times[f'Object {probability:.0%}', n] = generation_time

        print(f"\nHEURISTIC FUZZER - OBJECT {probability:.0%} - {n} INPUTS")
        print("Generated inputs:", len(inputs))
        for i, text in enumerate(inputs[:5], 1):
            print(f"Input {i}: {text}")

In [ ]:
# Heuristic grammar production coverage
heuristic_grammar_coverage = {}
for (probability, n), used in heuristic_used.items():
    heuristic_grammar_coverage[probability, n] = len(used & all_productions) / len(all_productions) * 100

    print(f"\nHEURISTIC GRAMMAR COVERAGE - OBJECT {probability:.0%} - {n} INPUTS")
    print("Total grammar productions:", len(all_productions))
    print("Used productions:", len(used & all_productions))
    print(f"Grammar coverage: {heuristic_grammar_coverage[probability, n]:.2f}%")
    print("Missing productions:", sorted(all_productions - used))

In [ ]:
# Heuristic parser code coverage
for (probability, n), inputs in heuristic_inputs.items():
    cov = coverage.Coverage(source=["json5kit"], branch=False, data_file=None)
    accepted = 0
    rejected = 0
    unexpected = 0

    cov.start()
    try:
        for test_input in inputs:
            try:
                json5kit.parse(test_input)
                accepted += 1
            except (ValueError, SyntaxError):
                rejected += 1
            except Exception as error:
                unexpected += 1
                print(type(error).__name__, error)
    finally:
        cov.stop()

    print("\nHEURISTIC FUZZER - JSON5KIT CODE COVERAGE")
    print(f"Object probability: {probability:.0%}")
    print("Input count:", n)
    code_coverage[f'Object {probability:.0%}', n] = cov.report()
    parser_results[f'Object {probability:.0%}', n] = (accepted, rejected, unexpected)

    print(f"\nTotal line coverage: {code_coverage[f'Object {probability:.0%}', n]:.2f}%")
    print("Accepted:", accepted)
    print("Rejected:", rejected)
    print("Unexpected errors:", unexpected)
    print(f"Acceptance rate: {accepted / n * 100:.2f}%")

In [ ]:
# Input complexity analysis
complexity_inputs = {}
for n, inputs in normal_inputs.items():
    complexity_inputs['Normal', n] = inputs
for (probability, n), inputs in heuristic_inputs.items():
    complexity_inputs[f'Object {probability:.0%}', n] = inputs

for (label, n), inputs in complexity_inputs.items():
    depths = []
    lengths = []
    for text in inputs:
        depth = 0
        largest = 0
        for char in text:
            if char == '{':
                depth += 1
                largest = max(largest, depth)
            elif char == '}':
                depth -= 1
        depths.append(largest)
        lengths.append(len(text))

    print(f"\n{label} - {n} INPUTS")
    print(f"Average object depth: {mean(depths):.2f}")
    print("Maximum object depth:", max(depths))
    print(f"Average input length: {mean(lengths):.2f}")

In [ ]:
# Mutation-based fuzzing
seed_inputs = [text for text in normal_inputs[INPUT_COUNTS[0]] if len(text) > 10][:20]
assert len(seed_inputs) == 20
mutation_inputs = {}

for minimum, maximum in MUTATION_SETTINGS:
    for n in INPUT_COUNTS:
        random.seed(SEED)
        mutation_fuzzer = MutationFuzzer(
            seed=seed_inputs, min_mutations=minimum, max_mutations=maximum
        )

        for _ in seed_inputs:
            mutation_fuzzer.fuzz()

        start = time.perf_counter()
        inputs = [mutation_fuzzer.fuzz() for _ in range(n)]
        generation_times[f'Mutation {minimum}-{maximum}', n] = time.perf_counter() - start
        mutation_inputs[minimum, maximum, n] = inputs

        print(f"\nMUTATION FUZZER - {minimum}-{maximum} - {n} INPUTS")
        print("Generated inputs:", len(inputs))
        for i, text in enumerate(inputs[:5], 1):
            print(f"Input {i}: {text!r}")

In [ ]:
# Mutation testing and json5kit code coverage
for (minimum, maximum, n), inputs in mutation_inputs.items():
    cov = coverage.Coverage(source=["json5kit"], branch=False, data_file=None)
    accepted = 0
    rejected = 0
    unexpected = 0

    cov.start()
    try:
        for test_input in inputs:
            try:
                json5kit.parse(test_input)
                accepted += 1
            except (ValueError, SyntaxError):
                rejected += 1
            except Exception as error:
                unexpected += 1
                print(type(error).__name__, error)
    finally:
        cov.stop()

    print("\nMUTATION FUZZER - JSON5KIT CODE COVERAGE")
    print(f"Mutations: {minimum}-{maximum}")
    print("Input count:", n)
    code_coverage[f'Mutation {minimum}-{maximum}', n] = cov.report()
    parser_results[f'Mutation {minimum}-{maximum}', n] = (accepted, rejected, unexpected)

    print(f"\nTotal line coverage: {code_coverage[f'Mutation {minimum}-{maximum}', n]:.2f}%")
    print("Accepted:", accepted)
    print("Rejected:", rejected)
    print("Unexpected errors:", unexpected)
    print(f"Acceptance rate: {accepted / n * 100:.2f}%")

In [ ]:
# json5kit parser execution time
input_groups = {}
for n, inputs in random_inputs.items():
    input_groups['Random', n] = inputs
for n, inputs in normal_inputs.items():
    input_groups['Normal', n] = inputs
for (probability, n), inputs in heuristic_inputs.items():
    input_groups[f'Object {probability:.0%}', n] = inputs
for (minimum, maximum, n), inputs in mutation_inputs.items():
    input_groups[f'Mutation {minimum}-{maximum}', n] = inputs

parse_times = {}
for (label, n), inputs in input_groups.items():
    times = []
    for _ in range(3):
        start = time.perf_counter()
        for text in inputs:
            try:
                json5kit.parse(text)
            except Exception:
                pass
        times.append(time.perf_counter() - start)

    parse_times[label, n] = median(times)
    print(f"{label} - {n} inputs: {parse_times[label, n]:.4f} seconds")

In [ ]:
# Input generation time
# Normal generation includes grammar tracking.
for (label, n), seconds in generation_times.items():
    print(f"{label} - {n} inputs: {seconds:.4f} seconds")

In [ ]:
# Final experimental results
for n in INPUT_COUNTS:
    print(f"\nFINAL RESULTS - {n} INPUTS")
    print("-" * 40)

    print("\nCODE COVERAGE")
    for (label, count), percentage in code_coverage.items():
        if count == n:
            print(f"{label}: {percentage:.2f}%")

    print("\nGRAMMAR PRODUCTION COVERAGE")
    print(f"Normal: {normal_grammar_coverage[n]:.2f}%")
    for probability in OBJECT_PROBABILITIES:
        print(f"Object {probability:.0%}: {heuristic_grammar_coverage[probability, n]:.2f}%")

    print("\nACCEPTANCE")
    for (label, count), (accepted, rejected, unexpected) in parser_results.items():
        if count == n:
            print(f"{label}: {accepted / n * 100:.2f}% | Accepted: {accepted} | Rejected: {rejected} | Unexpected: {unexpected}")

    print("\nTOTAL TESTING TIME")
    for (label, count), seconds in generation_times.items():
        if count == n:
            total = seconds + parse_times[label, n]
            print(f"{label}: {total:.4f} seconds")